In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arsalan9702/tickharm-pre-processed-audio")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio...
Path to dataset files: /kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio


In [49]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [48]:
!pip install -q torchlibrosa

In [50]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank
from torchlibrosa.augmentation import SpecAugmentation

class CNN14(nn.Module):
    def __init__(self, classes_num=4):
        super().__init__()

        self.spectrogram_extractor = Spectrogram(
            n_fft=1024, hop_length=320, win_length=1024,
            window='hann', center=True, pad_mode='reflect'
        )

        self.logmel_extractor = LogmelFilterBank(
            sr=16000, n_fft=1024, n_mels=64,
            fmin=50, fmax=8000, ref=1.0, amin=1e-10, top_db=None
        )

        self.spec_augmenter = SpecAugmentation(
            time_drop_width=64, time_stripes_num=2,
            freq_drop_width=8, freq_stripes_num=2
        )

        self.bn0 = nn.BatchNorm2d(64)

        self.conv_block1 = self._conv_block(1, 64)
        self.conv_block2 = self._conv_block(64, 128)
        self.conv_block3 = self._conv_block(128, 256)
        self.conv_block4 = self._conv_block(256, 512)

        self.fc1 = nn.Linear(512, 512)
        self.fc_out = nn.Linear(512, classes_num)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)
        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        x = torch.mean(x, dim=3)
        x = torch.mean(x, dim=2)

        x = F.relu(self.fc1(x))
        x = self.fc_out(x)

        return x

In [57]:
class TikHarmAudioDataset(Dataset):
    def __init__(self, root_dir, split, max_samples=16000*10):
        self.samples = []
        self.max_samples = max_samples

        classes = ["Adult Content", "Harmful Content", "Safe", "Suicide"]
        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for cls in classes:
            cls_path = os.path.join(root_dir, split, cls)
            for file in os.listdir(cls_path):
                if file.endswith(".wav"):
                    self.samples.append(
                        (os.path.join(cls_path, file), self.class_to_idx[cls])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, sr = torchaudio.load(path)

        waveform = waveform.mean(dim=0)

        if waveform.shape[0] < self.max_samples:
            pad = self.max_samples - waveform.shape[0]
            waveform = F.pad(waveform, (0, pad))
        else:
            waveform = waveform[:self.max_samples]

        return waveform, label

In [58]:
root = "/kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio/TikHarm_audio"

train_ds = TikHarmAudioDataset(root, "train")
val_ds   = TikHarmAudioDataset(root, "val")
test_ds  = TikHarmAudioDataset(root, "test")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))

2761 396 790


In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN14(classes_num=4)
model = model.to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

In [60]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [61]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    
    for inputs, labels in tqdm(loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
    return correct / total

In [63]:
best_val_acc = 0

for epoch in range(20):
    train_loss = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)
    scheduler.step()
    
    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Accuracy:", val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), "best_audio_cnn14.pth")
        else:
            torch.save(model.state_dict(), "best_audio_cnn14.pth")
        print("Best model saved.")

100%|██████████| 87/87 [00:19<00:00,  4.57it/s]


Epoch 1
Train Loss: 1.275331023095668
Val Accuracy: 0.42424242424242425
Best model saved.


100%|██████████| 87/87 [00:19<00:00,  4.58it/s]


Epoch 2
Train Loss: 1.225758492261514
Val Accuracy: 0.5227272727272727
Best model saved.


100%|██████████| 87/87 [00:18<00:00,  4.63it/s]


...

Epoch 18
Train Loss: 0.9983078919608017
Val Accuracy: 0.5782828282828283


100%|██████████| 87/87 [00:18<00:00,  4.59it/s]


Epoch 19
Train Loss: 0.9919290494644779
Val Accuracy: 0.5909090909090909


100%|██████████| 87/87 [00:18<00:00,  4.62it/s]


Epoch 20
Train Loss: 1.000420897171415
Val Accuracy: 0.5883838383838383


In [64]:
# Load best CNN14 audio model
best_audio_model = CNN14(classes_num=4)

state_dict = torch.load("best_audio_cnn14.pth", map_location=device)

best_audio_model.load_state_dict(state_dict)
best_audio_model = best_audio_model.to(device)

if torch.cuda.device_count() > 1:
    best_audio_model = nn.DataParallel(best_audio_model)

best_audio_model.eval()

# Evaluate on test set
test_acc = evaluate(best_audio_model, test_loader)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.6025316455696202
